In [1]:
import torch
from torch.nn.functional import sigmoid
from PIL import Image
from torchvision.models import resnet18, ResNet18_Weights
from torchvision.transforms.functional import to_pil_image

In [2]:
rn18 = resnet18(weights='DEFAULT')
rn18.eval()
tfm = ResNet18_Weights.IMAGENET1K_V1.transforms()


featmaps = {}

def build_save_featmap(name):
    def hook(module, args, output):
        if name in ['avgpool', 'fc']:
            return
        output = output[0]
        dim = list(range(1, output.ndim))
        mu = output.mean(dim=dim, keepdim=True)
        sigma = output.std(dim=dim, keepdim=True)
        output = ((output - mu)/(sigma + 1e-5) > 0).float()
        featmaps[name] = output
    return hook

for name, layer in rn18.named_children():
    layer.register_forward_hook(build_save_featmap(name))

In [3]:
# Copy the conv1 weights and detach them from autograd
conv1_weights = torch.clone(rn18.conv1.weight).detach()

# Scale the weights in range [0, 1] for visualization purposes
conv1_weights -= conv1_weights.min()
conv1_weights /= conv1_weights.max()

# Change order of channels for visualization purposes
conv1_weights = conv1_weights.permute(0, 1, 3, 2)

In [4]:
im = Image.open('images/waldek1.jpg')
t_input = tfm(im)

with torch.no_grad():
    out = rn18(t_input[None, ...])

In [5]:
def tensor_to_js(arr, name):
    lines = []
    lines.append(f'const {name} = [')

    for channel in arr:
        lines.append(' '*2 + '[')
        for row in channel:
            lines.append(' '*4 + '[' + r', '.join(fr'{x}' for x in row) + '],')
        lines.append(' '*2 + '],')

    lines.append('];')

    lines.append('')
    lines.append(f'export default {name};')
    return '\n'.join(lines)

In [9]:
from pathlib import Path

num_maps = {
    'conv1': 64,
    'layer1': 16,
    'layer2': 32,
    'layer3': 64,
    'layer4': 128,
}

for name, num in num_maps.items():
    t = featmaps[name][:num]
    text = tensor_to_js(t, name)
    Path(f'{name}.js').write_text(text)

    for i, ch in enumerate(t):
        ch_im = to_pil_image(ch)
        ch_im.save(f'activation_images/{name}_{i:02}.png')
        if name == 'conv1':
            weight_im = to_pil_image(conv1_weights[i])
            weight_im.save(f'activation_images/{name}_{i:02}_weight.png')

In [7]:
tfm.mean = [0, 0, 0]
tfm.std = [1, 1, 1]

tfm_im = tfm(im)

to_pil_image(tfm_im).save('activation_images/input.png')

zeros = torch.zeros_like(tfm_im[0])

to_pil_image(torch.stack([tfm_im[0], zeros, zeros])).save('activation_images/input_red.png')
to_pil_image(torch.stack([zeros, tfm_im[0], zeros])).save('activation_images/input_green.png')
to_pil_image(torch.stack([zeros, zeros, tfm_im[0]])).save('activation_images/input_blue.png')